# step2_F1_applied_all_GCM - Apply F1 to all GCM runs (global, 0.25 deg)

The **same** F1 (learned on EC-Earth3 historical -> ERA5, notebook step1) is applied to:
EC-Earth3 historical (for the validations), then the GCM 2025 / 2050 runs, hosing / no-hosing.

Result: "ERA5-like" GCMs at 0.25 deg. The **2025 no-hosing** run is the **target** of the RCM correction (notebook step3), where it is cropped to the RCM domain.

One file per run: `era5like_025/<run>_era5like_025.nc` (variables `pr`, `tasmax`, `tasmin`).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py, f1_downscaler.py: this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']
DOMAIN = bu.rcm_domain(cfg)   # RCM domain for regional diagnostics (None = global until the RCM path is filled in)

TRAIN_PERIOD = ("1981", "2014")     # must match the one used in notebook step1
APPLY_TO = ["gcm_historical", "gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
OVERWRITE = False
import f1_downscaler as f1
F1_DIR = bu.out_dir(cfg, "plan_B", "F1_downscaling", f"{TRAIN_PERIOD[0]}-{TRAIN_PERIOD[1]}")
OUT_DIR = bu.out_dir(cfg, "plan_B", "era5like_025")

## 1. Target grid = global ERA5 0.25 deg grid (the one F1 was trained on)

In [ ]:
era5_pr = bu.read_raw(cfg, "era5", "pr").isel(time=slice(0, 1))
tlat, tlon = era5_pr.lat.values, era5_pr.lon.values
print(f"Target grid: {len(tlat)} x {len(tlon)}")

## 2. Application

In [ ]:
applied = []
for name in APPLY_TO:
    out_path = OUT_DIR / f"{name}_era5like_025.nc"
    if out_path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); applied.append(name); continue
    if not all(bu.dataset_available(cfg, name, v) for v in VARS):
        print(f"[missing] {name} - fill in the paths in notebook 00"); continue
    print(f"-> {name}")
    sim = {v: bu.read_raw(cfg, name, v) for v in VARS}
    try:
        out = f1.apply_f1(sim, F1_DIR, tlat, tlon)
    except NotImplementedError as e:
        print("PLACEHOLDER:", e); break
    bu.save_ds(xr.Dataset({v: out[v] for v in VARS}), out_path)
    applied.append(name)

## 3. Quick checks

- grid identical to ERA5 0.25 deg; `pr >= 0`; `tasmin < tasmax`; no NaN;
- **is the GCM signal preserved?** change 2025 no-hosing - historical and hosing - no-hosing, monthly means over the RCM domain (global until the RCM path is filled in), **native GCM** vs **downscaled GCM**.
  An analogue method takes its values from ERA5: if the signal changes shape a lot, understand why before continuing.

In [ ]:
rows = []
for name in APPLY_TO:
    f = OUT_DIR / f"{name}_era5like_025.nc"
    if not f.exists():
        continue
    ds = xr.open_dataset(f, chunks={"time": 365})
    s = ds.isel(time=slice(0, 365 * 3))
    rows.append({"run": name,
                 "grid == ERA5": bool(bu.same_grid(ds["pr"], era5_pr)),
                 "pr min": float(s["pr"].min()),
                 "frac tasmin>=tasmax (should be 0)": float(((s["tasmin"] >= s["tasmax"]) & s["tasmax"].notnull()).mean()),
                 "frac NaN (pr)": float(s["pr"].isnull().mean())})
pd.DataFrame(rows)

In [ ]:
def monthly_native(name, var):
    return bu.area_mean(bu.read_raw(cfg, name, var), DOMAIN).groupby("time.month").mean().compute()

def monthly_ds(name, var):
    f = OUT_DIR / f"{name}_era5like_025.nc"
    return bu.area_mean(bu.read_processed(f, var), DOMAIN).groupby("time.month").mean().compute()

comparisons = [("gcm_2025_nohosing", "gcm_historical", "2025 no-hosing - historical"),
               ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - no-hosing")]
fig, axes = plt.subplots(len(comparisons), 3, figsize=(16, 7))
for r, (a, b, lab) in enumerate(comparisons):
    for c, var in enumerate(VARS):
        ax = axes[r, c]
        fa, fb = OUT_DIR / f"{a}_era5like_025.nc", OUT_DIR / f"{b}_era5like_025.nc"
        if not (fa.exists() and fb.exists()):
            ax.set_title(f"{lab} / {var}: missing"); continue
        raw = monthly_native(a, var) - monthly_native(b, var)
        ds_ = monthly_ds(a, var) - monthly_ds(b, var)
        ax.plot(raw.month, raw, "o-", label="native GCM"); ax.plot(ds_.month, ds_, "s--", label="after F1")
        ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{lab} - {var}"); ax.set_xlabel("month")
axes[0, 0].legend()
plt.tight_layout(); plt.savefig(bu.out_dir(cfg, "plan_B", "F1_downscaling") / "signal_before_after.png", dpi=110); plt.show()